# mIF Pipeline Python API Example

This notebook shows how to call the repo's Python API directly from a notebook. It is meant as a lightweight reference for stage-by-stage use, especially when you want to inspect results as Python dictionaries instead of driving the pipeline from the CLI.

It assumes `mif_pipeline` is already installed in the active environment, for example with `pip install -e .`.


In [ ]:
from pathlib import Path
import json

from mif_pipeline import (
    build_spatialdata,
    finalize_nimbus_multislide,
    load_config,
    merge_slide_ometiffs,
    qc_slide,
    run_instanseg,
    run_nimbus_chunked,
    run_nimbus_multislide,
    setup_slide,
)
from mif_pipeline.config import get_slide_config


In [ ]:
# Edit these for your own cluster or local run.
CONFIG_PATH = Path("~/codex/mIF-pipeline/prototyping/prototype_v2-Crop.yaml").expanduser()
SLIDE_ID = "SLIDE-0329_crop_2048"

config = load_config(CONFIG_PATH)
slide = get_slide_config(config, SLIDE_ID)

summary = {
    "config_path": str(CONFIG_PATH),
    "slide_id": SLIDE_ID,
    "slide_dir": slide["slide_dir"],
    "output_dir": slide["output_dir"],
    "seg_merge_path": slide["seg_merge"]["ome_path"],
    "full_merge_path": slide["full_merge"]["ome_path"],
    "mask_dir": slide["mask_export"]["mask_dir"],
    "nimbus_output_dir": slide["nimbus"]["output_dir"],
    "spatialdata_store": slide["spatialdata"]["store_path"],
}
summary


In [ ]:
def show_result(result):
    print(json.dumps(result, indent=2, default=str))


## Setup

In the current workflow, `setup` is usually run manually first to create or refresh the channel map. It is still useful to call through Python when you want to inspect the returned paths.


In [ ]:
setup_result = setup_slide(config, SLIDE_ID, dry_run=True)
show_result(setup_result)


## Stage Dry Runs

Each stage returns a small dictionary. Dry-run mode is a good way to inspect resolved paths and planned actions without touching any outputs.


In [ ]:
merge_plan = merge_slide_ometiffs(config, SLIDE_ID, dry_run=True)
instanseg_plan = run_instanseg(config, SLIDE_ID, dry_run=True)
nimbus_plan = run_nimbus_chunked(config, SLIDE_ID, dry_run=True)
multislide_nimbus_plan = run_nimbus_multislide(config, [SLIDE_ID], dry_run=True)
spatialdata_plan = build_spatialdata(config, SLIDE_ID, dry_run=True, return_sdata=False)
qc_result = qc_slide(config, SLIDE_ID)

show_result(
    {
        "merge": merge_plan,
        "instanseg": instanseg_plan,
        "nimbus": nimbus_plan,
        "nimbus_multislide": multislide_nimbus_plan,
        "spatialdata": spatialdata_plan,
        "qc": qc_result,
    }
)


## Real Stage Calls

Uncomment and run these one at a time when you want to execute the pipeline from Python. This is the same stage-by-stage pattern used by the CLI, just without the shell wrapper.


In [ ]:
# merge_result = merge_slide_ometiffs(config, SLIDE_ID, force=False)
# show_result(merge_result)

# instanseg_result = run_instanseg(config, SLIDE_ID, force=False)
# show_result(instanseg_result)

# nimbus_result = run_nimbus_chunked(config, SLIDE_ID, force=False)
# show_result(nimbus_result)

# multislide_chunk_result = run_nimbus_multislide(config, [SLIDE_ID], chunk_indices=[0], force=False)
# show_result(multislide_chunk_result)

# multislide_finalize_result = finalize_nimbus_multislide(config, [SLIDE_ID])
# show_result(multislide_finalize_result)

# spatialdata_result = build_spatialdata(config, SLIDE_ID, force=False, return_sdata=False)
# show_result(spatialdata_result)


## Notes

- `run_instanseg(...)` now writes the canonical whole-cell and nuclear mask TIFFs directly.
- Use `run_nimbus_multislide(..., chunk_indices=[...])` when you want to parallelize Nimbus over chunk indices.
- Run `finalize_nimbus_multislide(...)` after all chunk jobs finish to write the canonical global and per-slide tables.
- On IRIS, the shell scripts in `scripts/` are still the recommended operational path for multi-env runs.
- This notebook is mainly for direct Python use, debugging, and inspecting stage outputs in memory.
